# Load to Supabase

This notebook loads transformed data from `data/processed` into the final PostgreSQL/Supabase tables.

In [14]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2 import sql

## Path and Environment Configuration

In [15]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'processed').exists() else Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'
SCHEMA_SQL_PATH = PROJECT_ROOT / 'sql' / '01_schema_warehouse.sql'

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')

DB_URL = (
    os.getenv('SUPABASE_DB_URL')
    or os.getenv('DATABASE_URL')
    or os.getenv('POSTGRES_URL')
    or os.getenv('SUPABASE_POSTGRES_URL')
)

if not DB_URL:
    raise ValueError('Set SUPABASE_DB_URL in the .env file with the PostgreSQL connection string from Supabase.')

print('Project root:', PROJECT_ROOT.resolve())
print('Processed folder:', PROCESSED_DIR.resolve())
print('Schema SQL:', SCHEMA_SQL_PATH.resolve())

Project root: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse
Processed folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed
Schema SQL: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\sql\01_schema_warehouse.sql


## Load Table Configuration

Dimensions must be loaded first, followed by the fact table, because the fact table has foreign keys to the dimensions.

In [16]:
LOAD_TABLES = [
    'dim_channel_pembelian',
    'dim_lokasi',
    'dim_pelanggan',
    'dim_produk',
    'dim_status_transaksi',
    'dim_waktu',
    'fact_transaksi_layanan',
]

csv_inventory = []
for table_name in LOAD_TABLES:
    csv_path = PROCESSED_DIR / f'{table_name}.csv'
    csv_inventory.append({
        'table_name': table_name,
        'csv_path': str(csv_path),
        'exists': csv_path.exists(),
        'size_kb': round(csv_path.stat().st_size / 1024, 2) if csv_path.exists() else None,
    })

csv_inventory_df = pd.DataFrame(csv_inventory)
display(csv_inventory_df)

missing_files = csv_inventory_df.loc[~csv_inventory_df['exists'], 'csv_path'].tolist()
if missing_files:
    raise FileNotFoundError(f'Processed files are incomplete: {missing_files}')

,table_name,csv_path,exists,size_kb
0,dim_channel_pembelian,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,0.17
1,dim_lokasi,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,7.64
2,dim_pelanggan,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,62.17
3,dim_produk,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,2.61
4,dim_status_transaksi,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,0.13
5,dim_waktu,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,39.83
6,fact_transaksi_layanan,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...,True,1620.06


## Helper Database Load

In [17]:
def get_connection():
    return psycopg2.connect(DB_URL)


def execute_schema_sql(conn, schema_path):
    with open(schema_path, 'r', encoding='utf-8') as file:
        schema_sql = file.read()

    with conn.cursor() as cur:
        cur.execute(schema_sql)


def truncate_tables(conn, table_names, schema_name='public'):
    table_identifiers = [sql.Identifier(schema_name, table_name) for table_name in table_names]
    truncate_stmt = sql.SQL('TRUNCATE TABLE {} RESTART IDENTITY CASCADE').format(
        sql.SQL(', ').join(table_identifiers)
    )
    with conn.cursor() as cur:
        cur.execute(truncate_stmt)


def copy_csv_to_table(conn, csv_path, table_name, schema_name='public'):
    copy_stmt = sql.SQL(
        "COPY {} FROM STDIN WITH (FORMAT CSV, HEADER TRUE, NULL '')"
    ).format(sql.Identifier(schema_name, table_name))

    with conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(copy_stmt.as_string(conn), file)


def get_table_count(conn, table_name, schema_name='public'):
    count_stmt = sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(schema_name, table_name))
    with conn.cursor() as cur:
        cur.execute(count_stmt)
        return cur.fetchone()[0]


def get_database_columns(conn, table_name, schema_name='public'):
    query = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name = %s
        ORDER BY ordinal_position
    """
    with conn.cursor() as cur:
        cur.execute(query, (schema_name, table_name))
        return [row[0] for row in cur.fetchall()]

## Processed Data Preview Before Load

In [18]:
processed_preview = []

for table_name in LOAD_TABLES:
    csv_path = PROCESSED_DIR / f'{table_name}.csv'
    df_preview = pd.read_csv(csv_path, nrows=5)
    df_count = pd.read_csv(csv_path, usecols=[0])
    processed_preview.append({
        'table_name': table_name,
        'csv_row_count': len(df_count),
        'csv_column_count': len(df_preview.columns),
        'columns': ', '.join(df_preview.columns),
    })
    print(f'\n=== {table_name} ===')
    display(df_preview)

processed_preview_df = pd.DataFrame(processed_preview)
display(processed_preview_df)


=== dim_channel_pembelian ===


,id_channel,nama_channel,jenis_channel
0,1,MyTelkomsel,Digital
1,2,GraPARI,Fisik
2,3,Minimarket,Fisik
3,4,Website_Telkomsel,Digital
4,5,Mitra Dealer,Fisik



=== dim_lokasi ===


,id_lokasi,site_id,kelurahan,kecamatan,kabupaten_kota,provinsi,regional
0,1,SITE-0001,Menteng,Menteng,Jakarta Pusat,DKI Jakarta,Regional Jawa
1,2,SITE-0002,Gambir,Gambir,Jakarta Pusat,DKI Jakarta,Regional Jawa
2,3,SITE-0003,Tanah Abang,Tanah Abang,Jakarta Pusat,DKI Jakarta,Regional Jawa
3,4,SITE-0004,Kemayoran,Kemayoran,Jakarta Pusat,DKI Jakarta,Regional Jawa
4,5,SITE-0005,Senen,Senen,Jakarta Pusat,DKI Jakarta,Regional Jawa



=== dim_pelanggan ===


...


=== dim_waktu ===


,id_waktu,tanggal_lengkap,hari,bulan,kuartal,tahun
0,1,2022-01-01,Sabtu,Januari,Q1,2022
1,2,2022-01-02,Minggu,Januari,Q1,2022
2,3,2022-01-03,Senin,Januari,Q1,2022
3,4,2022-01-04,Selasa,Januari,Q1,2022
4,5,2022-01-05,Rabu,Januari,Q1,2022



=== fact_transaksi_layanan ===


,id_fakta,id_waktu,id_pelanggan,id_produk,id_lokasi,id_channel,id_status,nomor_transaksi,jumlah_pembelian,total_harga,jumlah_data_mb,durasi_telpon_menit,jumlah_sms
0,4,671,243,34,14,3,1,TRX006710000004,1,200000.0,44081.52,527,223
1,5,858,793,34,68,1,1,TRX008580000005,3,600000.0,12863.57,126,372
2,6,603,525,36,37,6,1,TRX006030000006,3,450000.0,37873.83,139,147
3,9,79,18,32,8,2,1,TRX000790000009,3,285000.0,14917.96,582,213
4,15,95,798,40,88,2,1,TRX000950000015,1,10000.0,27047.78,904,259


,table_name,csv_row_count,csv_column_count,columns
0,dim_channel_pembelian,6,3,"id_channel, nama_channel, jenis_channel"
1,dim_lokasi,98,7,"id_lokasi, site_id, kelurahan, kecamatan, kabu..."
2,dim_pelanggan,1000,7,"id_pelanggan, nomor_hp, nama_pelanggan, jenis_..."
3,dim_produk,50,6,"id_produk, kode_produk, nama_produk, jenis_pro..."
4,dim_status_transaksi,5,3,"id_status, kode_status, deskripsi_status"
5,dim_waktu,1096,6,"id_waktu, tanggal_lengkap, hari, bulan, kuarta..."
6,fact_transaksi_layanan,25364,13,"id_fakta, id_waktu, id_pelanggan, id_produk, i..."


## CSV Column vs Supabase Table Validation

This cell ensures CSV column names match database column names before the `COPY` process is executed.

In [19]:
column_validation = []

with get_connection() as conn:
    for table_name in LOAD_TABLES:
        csv_path = PROCESSED_DIR / f'{table_name}.csv'
        csv_columns = list(pd.read_csv(csv_path, nrows=0).columns)
        db_columns = get_database_columns(conn, table_name)

        column_validation.append({
            'table_name': table_name,
            'csv_columns': ', '.join(csv_columns),
            'db_columns': ', '.join(db_columns),
            'is_column_match': csv_columns == db_columns,
            'missing_in_db': ', '.join([column for column in csv_columns if column not in db_columns]),
            'missing_in_csv': ', '.join([column for column in db_columns if column not in csv_columns]),
        })

column_validation_df = pd.DataFrame(column_validation)
display(column_validation_df)

column_validation_df.to_csv(VALIDATION_DIR / 'load_column_validation.csv', index=False)

if not column_validation_df['is_column_match'].all():
    raise ValueError('Some CSV columns do not match the Supabase table structure. Check load_column_validation.csv.')

,table_name,csv_columns,db_columns,is_column_match,missing_in_db,missing_in_csv
0,dim_channel_pembelian,"id_channel, nama_channel, jenis_channel","id_channel, nama_channel, jenis_channel",True,,
1,dim_lokasi,"id_lokasi, site_id, kelurahan, kecamatan, kabu...","id_lokasi, site_id, kelurahan, kecamatan, kabu...",True,,
2,dim_pelanggan,"id_pelanggan, nomor_hp, nama_pelanggan, jenis_...","id_pelanggan, nomor_hp, nama_pelanggan, jenis_...",True,,
3,dim_produk,"id_produk, kode_produk, nama_produk, jenis_pro...","id_produk, kode_produk, nama_produk, jenis_pro...",True,,
4,dim_status_transaksi,"id_status, kode_status, deskripsi_status","id_status, kode_status, deskripsi_status",True,,
5,dim_waktu,"id_waktu, tanggal_lengkap, hari, bulan, kuarta...","id_waktu, tanggal_lengkap, hari, bulan, kuarta...",True,,
6,fact_transaksi_layanan,"id_fakta, id_waktu, id_pelanggan, id_produk, i...","id_fakta, id_waktu, id_pelanggan, id_produk, i...",True,,


## Load Data to Supabase

By default, this notebook does not rerun the schema SQL. If the tables have not been created yet, run `sql/01_schema_warehouse.sql` in the Supabase SQL Editor first, or change `RUN_SCHEMA_SQL = True`.

In [20]:
RUN_SCHEMA_SQL = False
TRUNCATE_BEFORE_LOAD = True

load_results = []

with get_connection() as conn:
    conn.autocommit = False
    try:
        if RUN_SCHEMA_SQL:
            print('Running schema SQL...')
            execute_schema_sql(conn, SCHEMA_SQL_PATH)

        if TRUNCATE_BEFORE_LOAD:
            print('Truncating target tables...')
            truncate_tables(conn, LOAD_TABLES)

        for table_name in LOAD_TABLES:
            csv_path = PROCESSED_DIR / f'{table_name}.csv'
            csv_row_count = len(pd.read_csv(csv_path, usecols=[0]))
            print(f'Loading {table_name} from {csv_path.name} ({csv_row_count:,} rows)...')
            copy_csv_to_table(conn, csv_path, table_name)
            db_row_count = get_table_count(conn, table_name)

            load_results.append({
                'table_name': table_name,
                'csv_row_count': csv_row_count,
                'db_row_count_after_load': db_row_count,
                'is_row_count_match': csv_row_count == db_row_count,
            })

        conn.commit()
        print('Load completed and the transaction has been committed.')
    except Exception:
        conn.rollback()
        print('Load failed. The transaction has been rolled back.')
        raise

load_results_df = pd.DataFrame(load_results)
display(load_results_df)
load_results_df.to_csv(VALIDATION_DIR / 'load_results.csv', index=False)

Truncating target tables...
Loading dim_channel_pembelian from dim_channel_pembelian.csv (6 rows)...
Loading dim_lokasi from dim_lokasi.csv (98 rows)...
Loading dim_pelanggan from dim_pelanggan.csv (1,000 rows)...
Loading dim_produk from dim_produk.csv (50 rows)...
Loading dim_status_transaksi from dim_status_transaksi.csv (5 rows)...
Loading dim_waktu from dim_waktu.csv (1,096 rows)...
Loading fact_transaksi_layanan from fact_transaksi_layanan.csv (25,364 rows)...
Load completed and the transaction has been committed.


,table_name,csv_row_count,db_row_count_after_load,is_row_count_match
0,dim_channel_pembelian,6,6,True
1,dim_lokasi,98,98,True
2,dim_pelanggan,1000,1000,True
3,dim_produk,50,50,True
4,dim_status_transaksi,5,5,True
5,dim_waktu,1096,1096,True
6,fact_transaksi_layanan,25364,25364,True


## Final Row Count Check

In [21]:
final_counts = []

with get_connection() as conn:
    for table_name in LOAD_TABLES:
        final_counts.append({
            'table_name': table_name,
            'row_count': get_table_count(conn, table_name),
        })

final_counts_df = pd.DataFrame(final_counts)
display(final_counts_df)
final_counts_df.to_csv(VALIDATION_DIR / 'load_final_counts.csv', index=False)

,table_name,row_count
0,dim_channel_pembelian,6
1,dim_lokasi,98
2,dim_pelanggan,1000
3,dim_produk,50
4,dim_status_transaksi,5
5,dim_waktu,1096
6,fact_transaksi_layanan,25364
